# TUT — preparación y clustering autocontenidos


## 2. Importaciones y rutas


In [1]:
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler

try:
    from IPython.display import display
except ImportError:
    display = print


# Si Jupyter se inicia fuera de la carpeta del paquete, escribe aquí su ruta.
# Ejemplo: PROJECT_ROOT = Path(r"C:/TFM/notebooks_autocontenidos_sin_core")
PROJECT_ROOT = None

cwd = Path.cwd().resolve()
root_candidates = [cwd, cwd.parent, cwd.parent.parent]
if PROJECT_ROOT is not None:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
else:
    ROOT = next(
        (
            candidate
            for candidate in root_candidates
            if sum((candidate / name).is_dir() for name in ["TUT", "TUJI1", "UJIIndoor", "SOD"])
            >= 2
        ),
        cwd,
    )

SEED = 42
TARGET_COLUMNS = ["TARGET_X_M", "TARGET_Y_M"]
print("Raíz utilizada:", ROOT)


Raíz utilizada: /home/coder/Indoor/Notebooks


## 3. Lectura y preprocesado RSSI

Se normalizan los nombres de columnas, se detectan automáticamente WAP/MAC
y el escalador se ajusta solo con la partición de entrenamiento.


In [2]:
def natural_key(text: str) -> List[object]:
    return [int(piece) if piece.isdigit() else piece for piece in re.split(r"(\d+)", text)]


def find_file_case_insensitive(filename: str, directories: Sequence[Path]) -> Path:
    """Busca un nombre sin depender de mayusculas/minusculas."""
    checked: List[str] = []
    target = filename.casefold()
    for directory in directories:
        directory = Path(directory)
        checked.append(str(directory / filename))
        if not directory.exists():
            continue
        direct = directory / filename
        if direct.exists():
            return direct.resolve()
        for child in directory.iterdir():
            if child.is_file() and child.name.casefold() == target:
                return child.resolve()
    raise FileNotFoundError(
        f"No se encontro {filename}. Rutas comprobadas:\n- " + "\n- ".join(checked)
    )


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [str(c).strip().upper() for c in out.columns]
    return out


def detect_rssi_columns(df: pd.DataFrame) -> List[str]:
    cols = [c for c in df.columns if re.fullmatch(r"(?:WAP|MAC)\d+", str(c).upper())]
    cols = sorted(cols, key=natural_key)
    if not cols:
        raise ValueError("No se detectaron columnas RSSI WAPnnn o MACnnn.")
    return cols


class RSSIPreprocessor:
    """Imputa ausencias, estandariza RSSI con train y anade mascara de deteccion."""

    def __init__(self, missing_value: float = 100.0, fill_value: float = -110.0, use_mask: bool = True):
        self.missing_value = float(missing_value)
        self.fill_value = float(fill_value)
        self.use_mask = bool(use_mask)
        self.scaler = StandardScaler()
        self.columns: List[str] = []

    def _clean(self, df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        raw = df[self.columns].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
        observed = np.isfinite(raw) & (raw != self.missing_value)
        clean = np.where(observed, raw, self.fill_value).astype(np.float32)
        return clean, observed.astype(np.float32)

    def fit(self, df: pd.DataFrame, columns: Sequence[str]) -> "RSSIPreprocessor":
        self.columns = list(columns)
        clean, _ = self._clean(df)
        self.scaler.fit(clean)
        return self

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        clean, mask = self._clean(df)
        scaled = self.scaler.transform(clean).astype(np.float32)
        if self.use_mask:
            return np.concatenate([scaled, mask], axis=1).astype(np.float32)
        return scaled

    def fit_transform(self, df: pd.DataFrame, columns: Sequence[str]) -> np.ndarray:
        return self.fit(df, columns).transform(df)


## 4. Coordenadas y división de posiciones


In [3]:
def position_key(df: pd.DataFrame, columns: Sequence[str]) -> pd.Series:
    return df[list(columns)].astype(str).agg("|".join, axis=1)


### 4.1. Adaptación específica de TUT


In [4]:
def _try_find(filename: str, directories: Sequence[Path]) -> Optional[Path]:
    try:
        return find_file_case_insensitive(filename, directories)
    except FileNotFoundError:
        return None


def _collapse_cluster_long(df: pd.DataFrame) -> pd.DataFrame:
    """Elimina la repeticion artificial de una fila para cada valor de K."""
    out = normalize_columns(df)
    out = out.drop(columns=["CLUSTER", "N_CLUSTERS", "SPLIT"], errors="ignore")
    if "ID" in out.columns and out["ID"].notna().all():
        out = out.drop_duplicates(subset=["ID"], keep="first")
    else:
        out = out.drop_duplicates(keep="first")
    return out.reset_index(drop=True)


def _add_local_metric_targets(
    df: pd.DataFrame,
    x_column: str = "POS_X",
    y_column: str = "POS_Y",
    crs_name: str = "LOCAL_METRES",
) -> pd.DataFrame:
    out = df.copy()
    out["TARGET_X_M"] = pd.to_numeric(out[x_column], errors="raise").astype(float)
    out["TARGET_Y_M"] = pd.to_numeric(out[y_column], errors="raise").astype(float)
    out["METRIC_CRS"] = crs_name
    return out


def _valid_stratification(
    values: pd.Series,
    n_total: int,
    test_size: float,
) -> Optional[pd.Series]:
    values = values.astype(str)
    counts = values.value_counts()
    n_test = int(np.ceil(n_total * test_size))
    n_train = n_total - n_test
    if (
        len(counts) > 1
        and counts.min() >= 2
        and len(counts) <= n_test
        and len(counts) <= n_train
    ):
        return values
    return None


def _split_tut_positions(
    data: pd.DataFrame,
    position_columns: Sequence[str],
    seed: int,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    work = data.copy()
    work["_POSITION_KEY"] = position_key(work, position_columns)
    positions = (
        work.groupby("_POSITION_KEY", as_index=False)
        .agg(
            CLIENT_ID=("CLIENT_ID", lambda s: s.mode().iloc[0]),
            FLOOR_LABEL=("FLOOR_LABEL", lambda s: s.mode().iloc[0]),
        )
    )
    positions["_DEVICE_FLOOR"] = (
        positions["CLIENT_ID"].astype(str) + "|" + positions["FLOOR_LABEL"].astype(str)
    )
    stratify = _valid_stratification(positions["_DEVICE_FLOOR"], len(positions), 0.30)
    if stratify is None:
        stratify = _valid_stratification(positions["FLOOR_LABEL"], len(positions), 0.30)
    train_keys, holdout_keys = train_test_split(
        positions["_POSITION_KEY"],
        test_size=0.30,
        random_state=seed,
        stratify=stratify,
    )

    holdout = positions[positions["_POSITION_KEY"].isin(set(holdout_keys))].copy()
    holdout_stratify = _valid_stratification(
        holdout["_DEVICE_FLOOR"], len(holdout), 0.50
    )
    if holdout_stratify is None:
        holdout_stratify = _valid_stratification(
            holdout["FLOOR_LABEL"], len(holdout), 0.50
        )
    val_keys, test_keys = train_test_split(
        holdout["_POSITION_KEY"],
        test_size=0.50,
        random_state=seed,
        stratify=holdout_stratify,
    )

    def select(keys: Sequence[str]) -> pd.DataFrame:
        return (
            work[work["_POSITION_KEY"].isin(set(keys))]
            .drop(columns="_POSITION_KEY")
            .reset_index(drop=True)
        )

    return select(train_keys), select(val_keys), select(test_keys)


## 5. Guardado y router RSSI→clúster

Los identificadores de fila permiten unir cada partición con sus rutas.
KMeans se ajusta con las posiciones de train; un Extra Trees aprende a
reproducir esas zonas desde RSSI para validación y test.


### 5.1. Guardado de las tres particiones


In [5]:
def _assign_row_ids(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    out = df.reset_index(drop=True).copy()
    out.insert(0, "ROW_ID", [f"{prefix}_{i:07d}" for i in range(len(out))])
    return out


def save_base_splits(
    train: pd.DataFrame,
    val: pd.DataFrame,
    test: pd.DataFrame,
    output_dir: Path,
    prefix: str,
) -> Dict[str, Path]:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    split_frames = {
        "train": _assign_row_ids(train, f"{prefix}_train"),
        "val": _assign_row_ids(val, f"{prefix}_val"),
        "test": _assign_row_ids(test, f"{prefix}_test"),
    }
    paths: Dict[str, Path] = {}
    for split, frame in split_frames.items():
        path = output_dir / f"{prefix}_{split}.csv"
        frame.to_csv(path, index=False)
        paths[split] = path
    return paths


def read_base_splits(output_dir: Path, prefix: str) -> Dict[str, pd.DataFrame]:
    output_dir = Path(output_dir)
    return {
        split: pd.read_csv(output_dir / f"{prefix}_{split}.csv")
        for split in ["train", "val", "test"]
    }


### 5.2. Entrenamiento del router y auditoría


In [6]:
def fit_rssi_routing(
    train: pd.DataFrame,
    val: pd.DataFrame,
    test: pd.DataFrame,
    rssi_columns: Sequence[str],
    k_values: Sequence[int],
    seed: int = SEED,
    n_estimators: int = 200,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Define zonas con XY de train y aprende RSSI -> zona para val/test.

    CLUSTER_ORACLE solo se conserva para diagnostico. La columna CLUSTER que
    consumen los modelos es predicha por RSSI en validacion y test.
    """
    pre = RSSIPreprocessor(use_mask=True).fit(train, rssi_columns)
    x_train = pre.transform(train)
    x_val = pre.transform(val)
    x_test = pre.transform(test)
    coords_train = train[TARGET_COLUMNS].to_numpy(dtype=float)
    unique_coords = np.unique(coords_train, axis=0)

    route_parts: List[pd.DataFrame] = []
    diagnostics: List[Dict[str, object]] = []
    for k in sorted(set(int(v) for v in k_values)):
        if k < 2 or k > len(unique_coords):
            continue
        kmeans = KMeans(n_clusters=k, random_state=seed, n_init=20)
        kmeans.fit(unique_coords)
        oracle = {
            "train": kmeans.predict(train[TARGET_COLUMNS].to_numpy(dtype=float)),
            "val": kmeans.predict(val[TARGET_COLUMNS].to_numpy(dtype=float)),
            "test": kmeans.predict(test[TARGET_COLUMNS].to_numpy(dtype=float)),
        }
        gate = ExtraTreesClassifier(
            n_estimators=n_estimators,
            min_samples_leaf=2,
            max_features="sqrt",
            class_weight="balanced",
            random_state=seed + k,
            n_jobs=-1,
        )
        gate.fit(x_train, oracle["train"])
        predicted = {
            "train": oracle["train"],
            "val": gate.predict(x_val),
            "test": gate.predict(x_test),
        }
        probabilities = {
            "train": np.ones(len(train), dtype=float),
            "val": np.max(gate.predict_proba(x_val), axis=1),
            "test": np.max(gate.predict_proba(x_test), axis=1),
        }
        for split, frame in [("train", train), ("val", val), ("test", test)]:
            route_parts.append(
                pd.DataFrame(
                    {
                        "ROW_ID": frame["ROW_ID"].astype(str).to_numpy(),
                        "SPLIT": split,
                        "N_CLUSTERS": k,
                        "CLUSTER": predicted[split].astype(int),
                        "CLUSTER_ORACLE": oracle[split].astype(int),
                        "GATE_CONFIDENCE": probabilities[split].astype(float),
                    }
                )
            )
        diagnostics.append(
            {
                "N_CLUSTERS": k,
                "VAL_GATE_ACCURACY": float(accuracy_score(oracle["val"], predicted["val"])),
                "TEST_GATE_ACCURACY_DIAGNOSTIC_ONLY": float(
                    accuracy_score(oracle["test"], predicted["test"])
                ),
                "VAL_MEAN_CONFIDENCE": float(np.mean(probabilities["val"])),
                "TEST_MEAN_CONFIDENCE": float(np.mean(probabilities["test"])),
                "N_TRAIN_POSITIONS": int(len(unique_coords)),
            }
        )
    if not route_parts:
        raise ValueError("No se genero ninguna configuracion de clustering.")
    return pd.concat(route_parts, ignore_index=True), pd.DataFrame(diagnostics)


def save_routing(
    routes: pd.DataFrame,
    diagnostics: pd.DataFrame,
    output_dir: Path,
    prefix: str,
) -> Tuple[Path, Path]:
    output_dir = Path(output_dir)
    route_path = output_dir / f"{prefix}_routes.csv"
    diagnostics_path = output_dir / f"{prefix}_routing_diagnostics.csv"
    routes.to_csv(route_path, index=False)
    diagnostics.to_csv(diagnostics_path, index=False)
    return route_path, diagnostics_path


def validate_splits(
    train: pd.DataFrame,
    val: pd.DataFrame,
    test: pd.DataFrame,
    position_columns: Sequence[str],
) -> Dict[str, object]:
    ids = [set(frame["ROW_ID"].astype(str)) for frame in [train, val, test]]
    if ids[0] & ids[1] or ids[0] & ids[2] or ids[1] & ids[2]:
        raise AssertionError("ROW_ID se solapa entre particiones.")
    pos = [set(position_key(frame, position_columns)) for frame in [train, val, test]]
    return {
        "rows": {"train": len(train), "val": len(val), "test": len(test)},
        "positions": {"train": len(pos[0]), "val": len(pos[1]), "test": len(pos[2])},
        "position_overlap": {
            "train_val": len(pos[0] & pos[1]),
            "train_test": len(pos[0] & pos[2]),
            "val_test": len(pos[1] & pos[2]),
        },
    }


## 6. Preparación completa de TUT


In [7]:
def prepare_tut_dataset(
    data_directories: Sequence[Path],
    output_dir: Path,
    k_values: Sequence[int] = tuple(range(2, 9)),
    seed: int = SEED,
    gate_estimators: int = 200,
    n_devices: int = 5,
) -> Dict[str, object]:
    """Prepara TUT con hold-out interno, agrupado por posicion completa."""
    directories = [Path(path) for path in data_directories]
    train_path = _try_find("Training_dataset.csv", directories)
    test_path = _try_find("Testing_dataset.csv", directories)
    source: Dict[str, object]
    if train_path is not None and test_path is not None:
        data = pd.concat(
            [pd.read_csv(train_path), pd.read_csv(test_path)], ignore_index=True
        )
        data = normalize_columns(data)
        source = {
            "method": "merged_original_training_and_testing_files",
            "training_file": str(train_path),
            "testing_file": str(test_path),
        }
    else:
        clusters_path = find_file_case_insensitive("TUT_clusters.csv", directories)
        data = _collapse_cluster_long(pd.read_csv(clusters_path))
        source = {
            "method": "collapsed_TUT_clusters_long_file",
            "clusters_file": str(clusters_path),
        }

    required = {"POS_X", "POS_Y", "FLOOR", "DEVICE"}
    missing = sorted(required - set(data.columns))
    if missing:
        raise ValueError(f"Faltan columnas TUT: {missing}")
    rssi = detect_rssi_columns(data)
    top_devices = data["DEVICE"].astype(str).value_counts().head(n_devices).index.tolist()
    data = data[data["DEVICE"].astype(str).isin(top_devices)].copy().reset_index(drop=True)
    data["CLIENT_ID"] = data["DEVICE"].astype(str)
    data["FLOOR_LABEL"] = pd.to_numeric(data["FLOOR"], errors="raise").astype(int)
    data = _add_local_metric_targets(data, crs_name="TUT_LOCAL_METRES")

    position_columns = ["FLOOR", "POS_X", "POS_Y"]
    train, val, test = _split_tut_positions(data, position_columns, seed)
    paths = save_base_splits(train, val, test, output_dir, "tut_top5")
    saved = read_base_splits(output_dir, "tut_top5")
    routes, route_diag = fit_rssi_routing(
        saved["train"], saved["val"], saved["test"], rssi, k_values, seed, gate_estimators
    )
    route_paths = save_routing(routes, route_diag, output_dir, "tut_top5")
    diagnostics = validate_splits(
        saved["train"], saved["val"], saved["test"], position_columns
    )
    diagnostics["clients"] = {
        split: sorted(frame["CLIENT_ID"].astype(str).unique().tolist())
        for split, frame in saved.items()
    }
    diagnostics["floors"] = {
        split: sorted(pd.to_numeric(frame["FLOOR_LABEL"]).astype(int).unique().tolist())
        for split, frame in saved.items()
    }
    diagnostics["top_devices"] = top_devices
    report = {
        "prefix": "tut_top5",
        "source": source,
        "files": {**{k: str(v) for k, v in paths.items()}, "routes": str(route_paths[0])},
        "diagnostics": diagnostics,
        "routing": route_diag.to_dict("records"),
    }
    output_dir = Path(output_dir)
    with (output_dir / "tut_top5_preparation_report.json").open("w", encoding="utf-8") as handle:
        json.dump(report, handle, indent=2, ensure_ascii=False)
    return report


## 7. Configuración y ejecución


In [8]:
DATA_DIRS = [
    Path.cwd(),
    ROOT / "TUT",
    ROOT / "Dataset",
    ROOT.parent,
    ROOT.parent / "Dataset",
    Path("/mnt/data"),
]
OUTPUT_DIR = ROOT / "prepared" / "TUT"

K_VALUES = list(range(2, 9))
SEED = 42
GATE_TREES = 200

report = prepare_tut_dataset(
    data_directories=DATA_DIRS,
    output_dir=OUTPUT_DIR,
    k_values=K_VALUES,
    seed=SEED,
    gate_estimators=GATE_TREES,
    n_devices=5,
)
PREFIX = report["prefix"]
print("Datos preparados en:", OUTPUT_DIR)
print("Prefijo:", PREFIX)


Datos preparados en: /home/coder/Indoor/Notebooks/prepared/TUT
Prefijo: tut_top5


## 8. Auditoría final


In [9]:
print("\n=== Auditoría de particiones ===")
display(pd.DataFrame([report["diagnostics"]["rows"]], index=["filas"]))
display(pd.DataFrame([report["diagnostics"]["positions"]], index=["posiciones"]))
display(pd.DataFrame([report["diagnostics"]["position_overlap"]], index=["solapamiento"]))
print("Dispositivos:", report["diagnostics"]["clients"])
print("Plantas:", report["diagnostics"]["floors"])
print("Top 5:", report["diagnostics"]["top_devices"])

print("\n=== Calidad del router RSSI ===")
display(pd.DataFrame(report["routing"]))

overlap = report["diagnostics"]["position_overlap"]
assert overlap["train_val"] == 0
assert overlap["train_test"] == 0
assert overlap["val_test"] == 0
print("\nOK: ninguna posición TUT aparece en más de una partición.")



=== Auditoría de particiones ===


,train,val,test
filas,1868,409,400


,train,val,test
posiciones,1839,394,395


,train_val,train_test,val_test
solapamiento,0,0,0


Dispositivos: {'train': ['LGE LG-D625', 'Letv x600', 'Sony E5823', 'samsung SM-A310F', 'samsung SM-A510F'], 'val': ['LGE LG-D625', 'Letv x600', 'Sony E5823', 'samsung SM-A310F', 'samsung SM-A510F'], 'test': ['LGE LG-D625', 'Letv x600', 'Sony E5823', 'samsung SM-A310F', 'samsung SM-A510F']}
Plantas: {'train': [0, 1, 2, 3, 4], 'val': [0, 1, 2, 3, 4], 'test': [0, 1, 2, 3, 4]}
Top 5: ['Letv x600', 'Sony E5823', 'samsung SM-A310F', 'LGE LG-D625', 'samsung SM-A510F']

=== Calidad del router RSSI ===


,N_CLUSTERS,VAL_GATE_ACCURACY,TEST_GATE_ACCURACY_DIAGNOSTIC_ONLY,VAL_MEAN_CONFIDENCE,TEST_MEAN_CONFIDENCE,N_TRAIN_POSITIONS
0,2,0.973105,0.9650,0.922096,0.937118,1839
1,3,0.933985,0.9425,0.865620,0.881965,1839
2,4,0.931540,0.9350,0.839155,0.866796,1839
3,5,0.897311,0.9025,0.804839,0.815543,1839
4,6,0.897311,0.9075,0.803521,0.824620,1839
5,7,0.882641,0.9025,0.768388,0.796975,1839
6,8,0.845966,0.8300,0.725345,0.738412,1839



OK: ninguna posición TUT aparece en más de una partición.
